# 03 — Association rules, phishing archetypes, surrogate tree

Every feature is already categorical, so frequent-itemset mining is a natural unsupervised counterpart to the classifiers. We look for rules of the form `SSLfinal_State=-1 AND URL_of_Anchor=-1 → phishing`, cluster the phishing subset with k-modes, and distill a depth-3 surrogate tree from a Random Forest's predictions.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from phishing.config import REPORTS_DIR
from phishing.data import load_xy
from phishing.mining import cluster_phishing, cluster_rule_crosstab, mine_rules, surrogate_tree
from phishing.models import build_model

pd.set_option("display.float_format", "{:.3f}".format)
X, y, _ = load_xy()

In [ ]:
rules_path = REPORTS_DIR / "association_rules.csv"
if rules_path.exists():
    rules = pd.read_csv(rules_path)
else:
    rules = mine_rules(X, y)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    rules.to_csv(rules_path, index=False)
print(f"{len(rules)} rules with phishing as the sole consequent")
rules.head(15)

In [ ]:
labels, centroids = cluster_phishing(X, y, n_clusters=3)
print("cluster sizes (phishing rows only):")
display(centroids)
xtab = cluster_rule_crosstab(X, labels, rules, top_n=6)
xtab.pivot(index="rule", columns="cluster", values="match_rate")

In [ ]:
rf = build_model("Random Forest")
rf.fit(X, y)
_, text = surrogate_tree(X, rf.predict(X), max_depth=3)
print(text)
(REPORTS_DIR / "surrogate_tree.txt").write_text(text)

The surrogate is fitted on the forest's **predictions**, not the labels, so it is an explanation of the model rather than a competing classifier. Combined with the high-lift rules it is the raw material for a human-readable "how to spot a phishing site" checklist.